### JAB-Hessian sensitivity estimation & adaptive precision allocation
# GPT-2, C4 calibration, dual-dataset metrics sweep

Derived from `version1_git_JAB_gpt2_notMod (1).ipynb` (same GPT-2 port, same GPTQ core, same
attention-aware joint loss, same greedy allocator -- sections 0-8 below are that notebook's
architecture/method stack, unchanged). This notebook changes what is measured and what it is
measured against:

1. **Calibration switches from WikiText-2 to C4** (`allenai/c4`, `en`, streaming `train` split) --
   the standard GPTQ/AWQ calibration protocol, so these numbers are comparable to published
   INT-quantization results. Calibration feeds the Hessian, the JAB scoring pass, and the
   fine-tuning batches alike.
2. **Evaluation runs on two disjoint corpora**: WikiText-2 test (as before) and a held-out C4
   `validation` slice -- disjoint from calibration by split, not by manual bookkeeping.
3. **Four metrics per arm per eval dataset**: perplexity, next-token top-1 accuracy (same
   strided windows/masking as perplexity, one forward pass), per-layer quantization error
   (`||W-What||_F/||W||_F` on `c_attn`, weight-space only), and per-layer attention
   reconstruction error (`||A-Ahat||_F/||A||_F` on the merged pre-`c_proj` attention output,
   teacher = float weights, student = quantized weights, same input captured from the live
   quantized model).
4. **One unified sweep** over `MODES = ["uniform", "jab", "joint", "adaptive_joint", "kl0"]` x
   `SWEEP_BITS = [3, 3.5, 4, 4.5, 6]`, replacing the three-criteria (JAB / JAB+propagation /
   oracle) sweep, the depth-scaling diagnostic, and the separate KL-ablation cell of the source
   notebook -- those are dropped entirely; only the plain JAB criterion remains, folded into the
   `jab` / `adaptive_joint` / `kl0` modes below.

No GPU here -- this notebook is written but not executed; see the accompanying note for expected
cell-by-cell behavior.

## Method (sections 0-8, unchanged from the source notebook)

0. GPT-2 architecture adapter -- 1. GPTQ core -- 2. Attention-aware joint loss --
3. Hutchinson trace estimator -- 4. Greedy sensitivity-per-cost allocator --
5. Model loading & per-layer helpers -- 6. Calibration/evaluation data (**C4, see above**) --
7. Validation (hand-written forward vs. real `GPT2LMHeadModel`, tiny-model FT check) --
8. The two pipeline passes, `pass_score` and `pass_quantize_eval` (**extended with an optional
`collect_metrics` flag for the two new weight/activation-space metrics**).

## Sweep (sections 9-13, new)

9. Setup (C4 calibration + both eval corpora + attention-recon eval chunks) --
10. fp32 control (both eval datasets) --
11. JAB-Hessian scoring, once, reused by every budget in the sweep --
12. The mode x bit-budget sweep itself --
13. Tidy long-form results table, printed and saved to `results.csv` /
`results_per_layer.json`.


In [1]:
!pip install -q transformers datasets

In [2]:
import csv
import gc
import itertools
import json
import math
import os
import random
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.autograd as autograd

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "gpt2"           # also: "gpt2-medium", "gpt2-large" -- NOT "gpt2-xl", see the
                             # GROUP_SIZE assertion in section 9 (1600 is not a multiple of 128)

# fp32 compute, fp64 GPTQ -- see the source notebook's section 0 comment for why (fp16 activations
# corrupt every Hessian and the quantized weight write-back; fp32 GPTQ compounds rounding into
# non-monotone perplexity vs. bit-width). Unchanged here.
GPU_DTYPE = torch.float32

# --- calibration (C4, not WikiText-2) ---
# Standard GPTQ/AWQ protocol: n_samples random contiguous seq_len-token spans, each drawn from a
# C4 document long enough to hold one, sampled with a fixed seed off the streaming `train` split.
C4_DATASET_NAME   = "allenai/c4"
C4_DATASET_CONFIG = "en"
CALIB_SEED        = 42
CALIB_N_SAMPLES   = 128
CALIB_SEQ_LEN     = 512
HESSIAN_N_BATCHES = 128     # batches used for H = 2 X^T X -- ALL of them, not a subset
HUTCH_SAMPLES     = 10      # Hutchinson probes per (layer, batch)
JAB_N_BATCHES     = 2       # batches averaged into each layer's trace
LAMBDA_KL         = 0.1     # 0.0 -> MSE-only ablation (also the "kl0" sweep mode below)
GROUP_SIZE        = 128     # must divide hidden_size (768/1024/1280 for gpt2/-medium/-large)

# --- GPTQ compute choices (unchanged from the source notebook) ---
GPTQ_DTYPE   = torch.float64
PERTURB_MODE = "rtn"        # "rtn" | "gptq" -- allocator's sensitivity-table heuristic only

# --- evaluation: WikiText-2 test AND a disjoint C4 validation slice ---
EVAL_MAX_LENGTH = 1024      # GPT-2's full context (n_positions) -- do not shrink this
EVAL_STRIDE     = 512
C4_EVAL_N_DOCS  = 2000      # validation-split documents concatenated into the C4 eval corpus --
                             # `validation` is a different stream than calibration's `train`, so
                             # this is disjoint from the calibration sample by construction

# --- attention-reconstruction metric ---
ATTN_RECON_N_CHUNKS = 8     # eval chunks averaged per (layer, eval dataset), at CALIB_SEQ_LEN so
                             # they share pass_quantize_eval's causal mask

# --- the sweep ---
SWEEP_BITS = [3, 3.5, 4, 4.5, 6]
MODES = ["uniform", "jab", "joint", "adaptive_joint", "kl0"]
# uniform/joint apply ONE flat bit-width to every layer -- there is no notion of a fractional
# "average" bits for a flat assignment, unlike jab/adaptive_joint/kl0's per-layer mixing. Rather
# than silently rounding 3.5/4.5 down and relabeling (which would just duplicate the 3-bit/4-bit
# row under a fake budget), those two modes are left undefined ("--") at fractional SWEEP_BITS
# entries -- see UNIFORM_AVAILABLE_WIDTHS in section 12.
UNIFORM_AVAILABLE_WIDTHS = {3, 4, 6}

# --- joint fine-tuning (unchanged from the source notebook) ---
JOINT_STEPS_PER_BLOCK = 200
JOINT_LR              = 1e-4   # unused by the FT block itself (see ALPHA_W/ALPHA_S) -- kept as
                                # the default for any other caller of pass_quantize_eval
JOINT_GRAD_CLIP       = 1.0
JOINT_LAYERS          = None   # e.g. range(0, 12, 2); others still get the GPTQ warm start
ALPHA_W               = 0.05   # lr_w = ALPHA_W * scale_flat.mean(); cosine-decayed
ALPHA_S               = 0.02   # lr_s = ALPHA_S * scale_flat.mean() -- LSQ scale lr
SCORE_EVERY           = 10     # score the held-out batch every N FT steps, not every step

# --- diagnostics ---
LOG_LOSS_COMPONENTS   = False        # print MSE / KL terms separately inside attention_loss


def free(*objs):
    """
    Run a collection and release cached VRAM.

    CAREFUL about what this does and does not do. The arguments are only there for readability:
    deleting them inside this function drops *this* frame's references, NOT the caller's bindings,
    so `free(x)` alone never releases `x`.
    """
    for o in objs:
        del o
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def vram(tag=""):
    if torch.cuda.is_available():
        a = torch.cuda.memory_allocated() / 1e9
        p = torch.cuda.max_memory_allocated() / 1e9
        print(f"    [vram{' ' + tag if tag else ''}: {a:.2f} GB now, {p:.2f} GB peak this pass]")


def reset_vram_peak():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()


print("Device:", DEVICE)
if DEVICE == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}, {props.total_memory / 1e9:.1f} GB")
print("Model:", MODEL_ID, "| compute dtype:", GPU_DTYPE, "| GPTQ dtype:", GPTQ_DTYPE)


Device: cuda
GPU: Tesla T4, 15.6 GB
Model: gpt2 | compute dtype: torch.float32 | GPTQ dtype: torch.float64


## 0. GPT-2 architecture adapter

In [3]:
class AttnConfig:
    """
    Shape bookkeeping pulled out of a HF GPT2Config.

    Deliberately the simplest case this pipeline handles: no RoPE (GPT-2 uses learned absolute
    position embeddings, added once at the embedding stage -- see `embed_cache` in section 5, not
    per layer), no GQA (num_kv_heads == num_heads always, so `n_rep` is always 1 and there is no
    `repeat_kv` anywhere in this notebook), no sliding window (plain causal mask).
    """

    def __init__(self, config):
        self.hidden_size   = config.n_embd
        self.num_heads     = config.n_head
        self.num_kv_heads  = config.n_head          # no GQA in GPT-2
        self.head_dim      = config.n_embd // config.n_head
        self.q_out         = self.num_heads * self.head_dim       # == hidden_size
        self.kv_out        = self.num_kv_heads * self.head_dim    # == hidden_size (no GQA shrink)
        self.n_rep         = self.num_heads // self.num_kv_heads  # always 1
        self.scaling       = self.head_dim ** -0.5
        self.n_layers      = config.n_layer

        # Q, K, V are equal thirds of the fused c_attn matrix -- unlike Mistral's GQA-driven split
        # into unequal Q vs. K/V chunks, this is exactly the GPT-2 c_attn.split(hidden_size, dim=1)
        # boundary, so reshape_weights/flatten_weights (section 2) need no GPT-2-specific logic.
        self.q_numel  = self.hidden_size * self.q_out
        self.kv_numel = self.hidden_size * self.kv_out

    @property
    def qkv_numel(self):
        return self.q_numel + 2 * self.kv_numel

    def __repr__(self):
        return (f"AttnConfig(hidden={self.hidden_size}, layers={self.n_layers}, "
                f"heads={self.num_heads}, head_dim={self.head_dim})")


def build_attn_mask(seq_len, device):
    """Plain causal mask -- GPT-2 has no sliding window."""
    return torch.ones(seq_len, seq_len, dtype=torch.bool, device=device).tril()

## 1. GPTQ core

In [4]:
def _quantize_to_grid(w_col, scale, bits):
    """
    Symmetric per-output-row fake quantization of a single input-column (shape: d_out) using a
    fixed per-row scale computed up front from the original weight statistics.
    """
    qmax = 2 ** (bits - 1) - 1
    return torch.clamp(torch.round(w_col / scale), -qmax, qmax) * scale


@torch.no_grad()
def gptq_quantize_layer(weight_in_out, H, bits=4, damp_percent=0.01, group_size=None,
                        act_order=True, return_scale=False, return_compact=False, work_dtype=None):
    """
    Quantizes a weight matrix in the (d_in, d_out) "x @ W" convention using GPTQ.
    UNCHANGED from the Mistral notebook -- this function is architecture-agnostic; it only ever
    sees a plain (d_in, d_out) matrix and a (d_in, d_in) Hessian.

    weight_in_out: (d_in, d_out) float tensor
    H: (d_in, d_in) Hessian
    damp_percent: Hessian damping for numerical stability (GPTQ default ~0.01)
    group_size: separate per-row scale per contiguous group of `group_size` input columns
    act_order: quantize columns in order of decreasing Hessian diagonal
    return_scale: ALSO return the exact per-(output channel, group) scale used, in the ORIGINAL
        (unpermuted) column order and the same (d_in, d_out) orientation -- so a caller can later
        fake-quantize the SAME weight onto the SAME grid (needed for the STE warm start).
    work_dtype: fp64 is GPTQ-canonical; GPT-2's Hessians are small enough (<=1280x1280) that this
        costs nothing, unlike the Mistral notebook's T4-driven fp32 compromise.

    Returns the fake-quantized weight (also written into weight_in_out in place), or
    (W_final, scale) if return_scale.
    """
    work_dtype = work_dtype or GPTQ_DTYPE
    device = weight_in_out.device
    W = weight_in_out.detach().clone().to(work_dtype).T.contiguous()  # (d_out, d_in)
    d_out, d_in = W.shape

    H = H.clone().to(work_dtype)
    mean_diag = H.diagonal().mean()
    H += damp_percent * mean_diag * torch.eye(d_in, dtype=work_dtype, device=device)

    if act_order:
        perm = torch.argsort(torch.diag(H), descending=True)
        invperm = torch.argsort(perm)
        W = W[:, perm]
        H = H[perm][:, perm]
    else:
        perm = invperm = torch.arange(d_in, device=device)

    H_inv = torch.linalg.inv(H)
    free(H)

    # Per-(row, group) scale, from the original weights in the permuted column order, before any
    # quantization.
    qmax = 2 ** (bits - 1) - 1
    gs = group_size if group_size is not None else d_in
    n_groups = (d_in + gs - 1) // gs
    scale = torch.zeros(d_out, n_groups, dtype=work_dtype, device=device)
    for g in range(n_groups):
        start, end = g * gs, min((g + 1) * gs, d_in)
        scale[:, g] = (W[:, start:end].abs().amax(dim=1) / qmax).clamp(min=1e-8)

    for i in range(d_in):
        row_scale = scale[:, i // gs]
        w_col = W[:, i]
        q_col = _quantize_to_grid(w_col, row_scale, bits)
        err = (w_col - q_col) / H_inv[i, i]
        if i + 1 < d_in:
            W[:, i + 1:] -= torch.outer(err, H_inv[i, i + 1:])
        W[:, i] = q_col
    free(H_inv)

    if act_order:
        W = W[:, invperm]

    W_final = W.T.contiguous().to(weight_in_out.dtype)  # back to (d_in, d_out)
    weight_in_out.copy_(W_final)

    if not return_scale:
        return W_final

    group_idx = torch.arange(d_in, device=device) // gs
    scale_cols = scale[:, group_idx]                                     # (d_out, d_in), permuted
    scale_cols = scale_cols[:, invperm] if act_order else scale_cols
    scale_expanded = scale_cols.T.contiguous().to(weight_in_out.dtype)

    if not return_compact:
        return W_final, scale_expanded

    # g_idx maps each ORIGINAL column to its group; groups are contiguous in PERMUTED order, so
    # this must go through invperm (original column c sits at permuted position invperm[c]).
    g_idx = (invperm // gs) if act_order else (torch.arange(d_in, device=device) // gs)
    return W_final, scale_expanded, scale.to(weight_in_out.dtype), g_idx


@torch.no_grad()
def gptq_quantize_conv1d(conv1d, H, bits=4, damp_percent=0.01, group_size=None,
                         act_order=True, return_scale=False, return_compact=False):
    """
    GPTQ for a HF `Conv1D` module (used by GPT-2's c_attn / c_proj / mlp.c_fc), whose weight is
    ALREADY `(d_in, d_out)` -- the forward is `x @ weight + bias`, unlike `nn.Linear`'s
    `(d_out, d_in)` / `x @ weight.T + bias`. This is the single most likely place to introduce a
    silent bug when porting GPTQ code between the two module conventions: get the orientation
    backwards and GPTQ quantizes along output channels instead of input channels, and nothing
    raises -- the shapes are square-ish enough (768x2304, 1024x4096, ...) that a transpose error
    would not even reliably crash, just silently degrade quantization quality.

    Guarded here with an explicit shape assertion (`gptq_quantize_layer` expects `weight_in_out`'s
    ROW count to equal `H`'s size, i.e. the input dimension) and re-checked in section 7's
    validation.

    Returns W_io (d_in, d_out) [, scale (d_in, d_out)].
    """
    d_in = H.shape[0]
    assert conv1d.weight.shape[0] == d_in, (
        f"Conv1D weight shape {tuple(conv1d.weight.shape)} does not put the input dimension "
        f"({d_in}) first -- if this fires, the (d_in, d_out) orientation assumption below is "
        f"wrong and gptq_quantize_layer would quantize along the wrong axis")
    # NO premature dtype cast here: gptq_quantize_layer upcasts to GPTQ_DTYPE (float64) itself.
    # An earlier version of this function cast to float32 HERE, before that upcast -- harmless in
    # isolation, but combined with a float16 model load (see GPU_DTYPE in section 0) it meant the
    # weight had already been rounded to fp16, then fp32, before ever reaching the float64 solve,
    # so the float64 work below was solving with already-corrupted inputs.
    W_io = conv1d.weight.data.clone()   # (d_in, d_out), NO transpose
    out = gptq_quantize_layer(W_io, H, bits=bits, damp_percent=damp_percent,
                              group_size=group_size, act_order=act_order,
                              return_scale=return_scale, return_compact=return_compact)
    if return_compact:
        W_q, scale, scale_compact, g_idx = out
        conv1d.weight.data.copy_(W_q.to(conv1d.weight.dtype))
        return W_q, scale, scale_compact, g_idx
    W_q, scale = out if return_scale else (out, None)
    conv1d.weight.data.copy_(W_q.to(conv1d.weight.dtype))
    return (W_q, scale) if return_scale else W_q


@torch.no_grad()
def quantize_qkv(c_attn, H, bits, group_size=None, damp_percent=0.01, return_scales=False, return_compact=False):
    """
    Applies GPTQ to GPT-2's fused c_attn matrix in ONE pass, then splits the result into the
    (W_Q, W_K, W_V) triple the rest of this notebook expects (see reshape_weights/flatten_weights
    in section 2). This is the Mistral notebook's three-separate-nn.Linear
    `quantize_qkv(projs, H, ...)` collapsed into one call, and it is mathematically identical to
    running GPTQ on Q, K, V separately with the same shared Hessian H: GPTQ's sequential
    input-column processing and error compensation never look at the output dimension, and the
    per-(output-row, group) scale is already computed independently per output row -- so grouping
    2304 output columns into one matrix vs. three 768-column matrices changes nothing about what
    gets computed, only how many Python-level calls it takes.
    """
    group_size = GROUP_SIZE if group_size is None else group_size
    out = gptq_quantize_conv1d(c_attn, H, bits=bits, damp_percent=damp_percent,
                               group_size=group_size, act_order=True, return_scale=return_scales,
                               return_compact=return_compact)
    if not return_scales:
        return out   # already applied in place; caller discards this when want_scales is False

    if return_compact:
        W_full, scale_full, scale_compact, g_idx = out
    else:
        W_full, scale_full = out
    hs = W_full.shape[0]                        # d_in == hidden_size; d_out == 3*hidden_size
    ws     = [W_full[:, :hs],     W_full[:, hs:2 * hs],     W_full[:, 2 * hs:]]
    scales = [scale_full[:, :hs], scale_full[:, hs:2 * hs], scale_full[:, 2 * hs:]]
    if return_compact:
        return ws, scales, scale_compact, g_idx   # scale_compact (3*hs, n_groups) stacked Q/K/V
    return ws, scales


@torch.no_grad()
def grid_perturbation(W_io, bits, group_size=None, H=None, mode=None):
    """
    HAWQ-V2's `||Q(W) - W||_F^2`, measured in weight space on GPTQ's own per-(output channel,
    group) symmetric grid. Unchanged from the Mistral notebook.

    mode="rtn"  : round-to-nearest on that grid. Milliseconds.
    mode="gptq" : the full GPTQ solve (requires H). Reproduces the GPT-2 notebook exactly, much
                  slower -- GPT-2's small matrices make this far more affordable than on Mistral,
                  but "rtn" stays the default for parity.
    """
    mode = PERTURB_MODE if mode is None else mode
    if mode == "gptq":
        if H is None:
            raise ValueError('mode="gptq" needs the Hessian H')
        W_q = gptq_quantize_layer(W_io.clone(), H, bits=bits, group_size=group_size, act_order=True)
        out = (W_q - W_io).pow(2).sum().item()
        free(W_q)
        return out

    W = W_io.T                                   # (d_out, d_in)
    d_out, d_in = W.shape
    gs = group_size if group_size is not None else d_in
    qmax = 2 ** (bits - 1) - 1
    total = 0.0
    for start in range(0, d_in, gs):
        blk = W[:, start:min(start + gs, d_in)].to(torch.float32)
        scale = (blk.abs().amax(dim=1, keepdim=True) / qmax).clamp(min=1e-8)
        blk_q = torch.clamp(torch.round(blk / scale), -qmax, qmax) * scale
        total += (blk_q - blk).pow(2).sum().item()
    return total

## 2. Attention-aware joint loss

In [5]:
def reshape_weights(w_flat, cfg):
    """
    Flat vector -> Q, K, V in the (d_in, d_out) "x @ W" convention.

    Unchanged from the Mistral notebook -- it already handled the general case. For GPT-2,
    q_out == kv_out == hidden_size (no GQA), so the three slices are always equal thirds, exactly
    matching `c_attn`'s own `Q, K, V = c_attn(x).split(hidden_size, dim=2)` boundary.
    """
    h, q_n, kv_n = cfg.hidden_size, cfg.q_numel, cfg.kv_numel
    return (w_flat[0:q_n].reshape(h, cfg.q_out),
            w_flat[q_n:q_n + kv_n].reshape(h, cfg.kv_out),
            w_flat[q_n + kv_n:q_n + 2 * kv_n].reshape(h, cfg.kv_out))


def flatten_weights(W_Q, W_K, W_V):
    """Inverse of reshape_weights (all three in (d_in, d_out) orientation)."""
    return torch.cat([W_Q.reshape(-1), W_K.reshape(-1), W_V.reshape(-1)])


def compute_attention(W_Q, W_K, W_V, X, cfg, b_Q=None, b_K=None, b_V=None, attn_mask=None):
    """
    Attention output and attention weights, matching HF's GPT2Attention (eager path) exactly.

    No RoPE (position info already lives in X via the embedding-stage `wpe` addition -- see
    `embed_cache` in section 5) and no GQA expansion (num_kv_heads == num_heads always), which is
    why this is shorter than the Mistral notebook's version: there is no cos/sin cache to thread
    through, and no `repeat_kv` call.

    W_Q: (hidden, hidden); W_K/W_V: (hidden, hidden) -- all three equal-sized, unlike Mistral's GQA
        split. b_Q/b_K/b_V: GPT-2's c_attn bias, split the same way as the weight; always present
        for the real model, always fixed (never a GPTQ/STE target).
    X: (batch, seq_len, hidden)

    Returns A_hat (batch, seq_len, hidden) -- heads merged, pre-c_proj -- and attn_weights
    (batch, num_heads, seq_len, seq_len), post-softmax.
    """
    B, T, _ = X.shape

    Q = X @ W_Q + (b_Q if b_Q is not None else 0)
    K = X @ W_K + (b_K if b_K is not None else 0)
    V = X @ W_V + (b_V if b_V is not None else 0)

    Q = Q.view(B, T, cfg.num_heads,    cfg.head_dim).transpose(1, 2)   # (B, 12, T, 64)
    K = K.view(B, T, cfg.num_kv_heads, cfg.head_dim).transpose(1, 2)   # (B, 12, T, 64)
    V = V.view(B, T, cfg.num_kv_heads, cfg.head_dim).transpose(1, 2)

    scores = (Q @ K.transpose(-2, -1)) * cfg.scaling

    if attn_mask is None:
        attn_mask = build_attn_mask(T, X.device)
    scores = scores.masked_fill(~attn_mask, float("-inf"))

    attn_weights = torch.softmax(scores, dim=-1, dtype=torch.float32).to(Q.dtype)

    A_hat = (attn_weights @ V).transpose(1, 2).contiguous().view(B, T, cfg.q_out)
    return A_hat, attn_weights


def mse_loss(w_flat, X, target_A, cfg, b_Q=None, b_K=None, b_V=None, attn_mask=None):
    """L_mse = ||A(X) - A_hat(X)||^2 -- the primary loss."""
    W_Q, W_K, W_V = reshape_weights(w_flat, cfg)
    A_hat, _ = compute_attention(W_Q, W_K, W_V, X, cfg, b_Q=b_Q, b_K=b_K, b_V=b_V,
                                 attn_mask=attn_mask)
    return F.mse_loss(A_hat, target_A)


def kl_loss(w_flat, X, target_attn, cfg, b_Q=None, b_K=None, b_V=None, attn_mask=None):
    """L_kl = KL(target attention maps || predicted attention maps)."""
    W_Q, W_K, W_V = reshape_weights(w_flat, cfg)
    _, attn_weights = compute_attention(W_Q, W_K, W_V, X, cfg, b_Q=b_Q, b_K=b_K, b_V=b_V,
                                        attn_mask=attn_mask)
    # KL(P || Q) = sum(P * log(P/Q)); P = target_attn, Q = attn_weights.
    # Masked entries have P = 0 and contribute 0 (F.kl_div uses xlogy).
    log_q = torch.log(attn_weights + 1e-8)  # epsilon for stability
    return F.kl_div(log_q, target_attn, reduction="batchmean")  # as in Q-BERT & APTQ


def attention_loss(w_flat, X, target_A, cfg, target_attn=None, lambda_kl=0.1,
                   b_Q=None, b_K=None, b_V=None, attn_mask=None, log_components=False):
    """MSE only if target_attn is None or lambda_kl == 0; otherwise MSE + lambda_kl * KL.

    log_components: if True, print the MSE and (raw, weighted) KL terms separately. Diagnostic
        only -- does not change what's returned or optimized. Useful because F.kl_div's
        reduction="batchmean" divides only by batch size while F.mse_loss averages over every
        element, so the raw KL term can run orders of magnitude above lambda_kl * KL's nominal
        weight; this makes that visible without changing the reduction.
    """
    mse = mse_loss(w_flat, X, target_A, cfg, b_Q=b_Q, b_K=b_K, b_V=b_V, attn_mask=attn_mask)
    loss = mse
    kl = None
    if target_attn is not None and lambda_kl:
        kl = kl_loss(w_flat, X, target_attn, cfg, b_Q=b_Q, b_K=b_K, b_V=b_V, attn_mask=attn_mask)
        loss = loss + lambda_kl * kl
    if log_components:
        kl_str = (f", kl={kl.item():.6f}, lambda_kl*kl={lambda_kl * kl.item():.6f}"
                  if kl is not None else ", kl=n/a")
        print(f"    [attention_loss] mse={mse.item():.6f}{kl_str}")
    return loss

## 3. Hutchinson trace estimator

In [6]:
def hessian_vector_product(loss_fn, params, vector, retain_graph=True):
    # create_graph=True keeps the graph alive for the second grad -- do not release it
    grad = autograd.grad(loss_fn(params), params, create_graph=True, retain_graph=True)[0]
    hvp = autograd.grad(grad, params, grad_outputs=vector, retain_graph=True)[0]
    del grad
    return hvp


def hutchinson_trace_estimator(loss_fn, params, samples=50):
    # trace(H) ~= (1/n) * sum(v_i^T H v_i)
    if not params.requires_grad:
        params.requires_grad_(True)

    estimated_trace = 0.0
    for _ in range(samples):
        # Rademacher vector (as in HAWQ-V2)
        vec = (torch.randint(0, 2, params.shape, device=params.device) * 2 - 1).to(params.dtype)
        hvp = hessian_vector_product(loss_fn, params, vec)
        estimated_trace += torch.dot(vec.flatten(), hvp.flatten())
        del vec, hvp
    return estimated_trace / samples

## 4. Greedy sensitivity-per-cost allocator

In [7]:
BIT_WIDTHS = [2, 3, 4, 8, 16]


def cost(bits, param_count=1.0):
    """
    Memory cost of one unit at a given bit-width. Every GPT-2 layer holds the same number of
    Q/K/V parameters, so param_count=1.0 keeps `budget` directly readable as "average bits per
    layer", exactly as in the Mistral notebook.
    """
    return bits * param_count


def greedy_allocate(scores, budget):
    """
    scores: {unit_name: {bits: sensitivity}};  budget: max total cost.
      1. Start every unit at the HIGHEST bit-width.
      2. While over budget, apply the downgrade with the best cost-saved-per-accuracy-lost ratio.
      3. Spend leftover budget on the best affordable upgrades.
    """
    current_bits = {name: max(BIT_WIDTHS) for name in scores}

    def total_cost():
        return sum(cost(current_bits[n]) for n in scores)

    def total_sensitivity():
        return sum(scores[n][current_bits[n]] for n in scores)

    while total_cost() > budget:                     # downgrade loop
        best = best_bits = best_ratio = None
        for name in scores:
            lower = [b for b in BIT_WIDTHS if b < current_bits[name]]
            if not lower:
                continue                             # already at the lowest bit-width
            nb = max(lower)
            saved = cost(current_bits[name]) - cost(nb)
            added = scores[name][nb] - scores[name][current_bits[name]]
            ratio = added / saved
            if best_ratio is None or ratio < best_ratio:
                best_ratio, best, best_bits = ratio, name, nb
        if best is None:
            break                                    # nothing left to downgrade
        current_bits[best] = best_bits

    made_an_upgrade = True                           # spend any remaining budget
    while made_an_upgrade:
        made_an_upgrade = False
        best = best_bits = best_ratio = None
        for name in scores:
            higher = [b for b in BIT_WIDTHS if b > current_bits[name]]
            if not higher:
                continue
            nb = min(higher)
            extra = cost(nb) - cost(current_bits[name])
            if total_cost() + extra > budget:
                continue                             # can't afford it
            ratio = (scores[name][current_bits[name]] - scores[name][nb]) / extra
            if best_ratio is None or ratio > best_ratio:
                best_ratio, best, best_bits = ratio, name, nb
        if best is not None:
            current_bits[best] = best_bits
            made_an_upgrade = True

    return current_bits, total_cost(), total_sensitivity()


def brute_force_optimal(scores, budget, max_units=8):
    """
    Exhaustive search, for sanity-checking the greedy allocator on a small subset.
    Guarded: with 5 bit-widths this is 5^n_units combinations.
    """
    names = list(scores.keys())
    if len(names) > max_units:
        raise ValueError(f"brute_force_optimal over {len(names)} units = "
                         f"{len(BIT_WIDTHS)}^{len(names)} combinations. Restrict `scores` to at "
                         f"most {max_units} units, or raise max_units if you really mean it.")
    best_assignment = best_cost = None
    best_sensitivity = float("inf")
    for combo in itertools.product(*([BIT_WIDTHS] * len(names))):
        c = sum(cost(b) for b in combo)
        if c > budget:
            continue
        s = sum(scores[names[i]][combo[i]] for i in range(len(names)))
        if s < best_sensitivity:
            best_sensitivity, best_cost = s, c
            best_assignment = dict(zip(names, combo))
    return best_assignment, best_cost, best_sensitivity


def scores_to_allocator_format(jab_scores, bit_widths=None):
    """Heuristic table (trace / bits^1.5), kept as an alternative to the HAWQ-V2 measured form."""
    bit_widths = BIT_WIDTHS if bit_widths is None else bit_widths
    return {n: {b: t / (b ** 1.5) for b in bit_widths} for n, t in jab_scores.items()}


def layer_name(i):
    return f"layer_{i}_QKV"


def layer_idx_of(name):
    return int(name.split("_")[1])

## 5. Model loading & per-layer helpers

In [8]:
from transformers import GPT2LMHeadModel, GPT2TokenizerFast, AutoConfig
from datasets import load_dataset


def load_model_and_tokenizer(model_id, device):
    """
    Loads a full, resident GPT-2 -- fits whole in VRAM/RAM (124 MB to 1.5 GB depending on
    `MODEL_ID`), so there is no need to stream shards off disk one tensor at a time.

    Every arm below (sections 10-15) calls this itself, once per arm, and gets back a BRAND NEW
    model. This matters: `pass_quantize_eval` quantizes `c_attn.weight` IN PLACE. Reusing one
    resident model across arms -- as an earlier version of this notebook did, loading `reader`
    once in section 9 and never again -- means each arm's damage compounds into the next: the
    adaptive-allocation scoring pass would run its Hessians against an already-4-bit model instead
    of the intended float baseline, and every arm after the first quantizing one would be
    GPTQ-quantizing an already-quantized matrix. The working reference notebook avoids this by
    calling `AutoModelForCausalLM.from_pretrained("gpt2")` fresh before every single arm; this
    function is that same fresh-load, just factored out.

    `pass_score`/`pass_quantize_eval`'s `reader` parameter is simply bound to whichever resident
    model this call returns -- `load_decoder_layer` indexes into it instead of allocating from a
    shard, which is what lets those two functions' signatures and per-layer loop structure stay
    unchanged regardless of which arm is calling them.
    """
    tokenizer = GPT2TokenizerFast.from_pretrained(model_id)
    tokenizer.pad_token = tokenizer.eos_token
    # Plain fp32, unconditionally -- see the GPU_DTYPE comment in section 0 for why.
    model = GPT2LMHeadModel.from_pretrained(model_id).to(device).to(GPU_DTYPE)
    model.config._attn_implementation = "eager"
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)
    return model, tokenizer


def load_decoder_layer(reader, config, idx):
    """
    No allocation, no meta-device trick -- `reader` is the resident model, so this is just an
    index into `reader.transformer.h`. Kept as a function purely so `pass_score`/
    `pass_quantize_eval`'s per-layer loop bodies read identically regardless of what `reader` is
    bound to.
    """
    return reader.transformer.h[idx]


def qkv_of(layer):
    """GPT-2 has one fused QKV module, not three separate projections."""
    return layer.attn.c_attn


def qkv_weights_io(layer, dtype=None):
    """
    The layer's Q/K/V in the (d_in, d_out) convention, split out of the fused c_attn matrix.
    `c_attn.weight` is already (d_in, d_out) = (hidden, 3*hidden) -- Conv1D, not nn.Linear, so
    there is NO transpose here. Getting this backwards is exactly the bug `gptq_quantize_conv1d`
    (section 1) guards against.
    """
    W = layer.attn.c_attn.weight
    hs = W.shape[0]
    W_Q, W_K, W_V = W[:, :hs], W[:, hs:2 * hs], W[:, 2 * hs:]
    if dtype:
        return W_Q.to(dtype), W_K.to(dtype), W_V.to(dtype)
    return W_Q, W_K, W_V


def qkv_bias_io(layer, dtype=None):
    """
    GPT-2's c_attn bias, split the same way as the weight. Biases are NEVER a GPTQ/STE target --
    callers snapshot them once per layer (alongside the float weight snapshot) and pass the same
    fixed b_Q/b_K/b_V into every compute_attention call for that layer, teacher and student alike.
    """
    b = layer.attn.c_attn.bias
    hs = b.shape[0] // 3
    b_Q, b_K, b_V = b[:hs], b[hs:2 * hs], b[2 * hs:]
    if dtype:
        return b_Q.to(dtype), b_K.to(dtype), b_V.to(dtype)
    return b_Q, b_K, b_V


@torch.no_grad()
def write_qkv_flat(layer, w_flat, cfg):
    """Scatter a flat [W_Q|W_K|W_V] vector back into c_attn's weight, in place, in the same
    (d_in, d_out) orientation it's already stored in -- no transpose."""
    W_Q, W_K, W_V = reshape_weights(w_flat, cfg)
    hs = cfg.hidden_size
    W = layer.attn.c_attn.weight.data
    W[:, :hs].copy_(W_Q.to(W.dtype))
    W[:, hs:2 * hs].copy_(W_K.to(W.dtype))
    W[:, 2 * hs:].copy_(W_V.to(W.dtype))


def block_forward(layer, h, cfg, attn_mask, w_qkv=None, return_attn=False):
    """
    One GPT-2 block, run manually for the QKV/attention piece (the part that needs to accept a
    swapped-in STE-quantized weight) but delegating everything else -- both LayerNorms, c_proj,
    the MLP -- to the real resident modules. Used by section 7's validation (to prove this
    hand-written piece matches the real model) and nowhere else: the main pipeline (section 8)
    gets its activations from real forward passes via hooks (`collect_hessian_via_hook`,
    `capture_c_attn_input` below), not from a hand-propagated hidden-state cache, so there is no
    second place this logic could silently drift from the real model's own forward.

    w_qkv: optional (W_Q, W_K, W_V) in (d_in, d_out) form, used INSTEAD of the module's own c_attn
        weight.
    """
    x = layer.ln_1(h)
    b_Q, b_K, b_V = qkv_bias_io(layer)
    if w_qkv is not None:
        W_Q, W_K, W_V = w_qkv
    else:
        W_Q, W_K, W_V = qkv_weights_io(layer)
    A, attn_w = compute_attention(W_Q, W_K, W_V, x, cfg, b_Q=b_Q, b_K=b_K, b_V=b_V,
                                  attn_mask=attn_mask)
    attn_out = layer.attn.c_proj(A.to(layer.attn.c_proj.weight.dtype))
    h = h + attn_out
    h = h + layer.mlp(layer.ln_2(h))
    return (h, attn_w) if return_attn else (h, None)


@torch.no_grad()
def embed_cache(reader, ids_list, device=None, dtype=None):
    """Hidden states entering layer 0, for every sequence -- used only by section 7's validation.
    `wte(ids) + wpe(positions)`: GPT-2 bakes ALL position information in once, here; there is no
    per-layer RoPE cache anywhere in this notebook."""
    device = device or DEVICE
    dtype = dtype or GPU_DTYPE
    wte = reader.transformer.wte.weight.to(device=device, dtype=dtype)
    wpe = reader.transformer.wpe.weight.to(device=device, dtype=dtype)
    out = []
    for ids in ids_list:
        ids = ids.to(device)
        T = ids.shape[1]
        pos = torch.arange(T, device=device)
        out.append(F.embedding(ids, wte) + F.embedding(pos, wpe)[None])
    free(wte, wpe)
    return out


def collect_hessian_via_hook(model, module, calibration_batches, device):
    """
    Registers a forward PRE-hook on `module` (a block's `attn.c_attn`), runs `calibration_batches`
    through the WHOLE model in no_grad mode, and returns the accumulated Hessian H = 2 X^T X for
    that layer. Ported verbatim (module/variable names aside) from the working reference notebook.

    This replaces hand-computing `X = layer.ln_1(h)` against a manually-propagated hidden-state
    cache. The two should be mathematically identical -- section 7's validation confirms the
    hand-written forward matches the real model to within float precision -- but this is the
    version actually exercised end to end by the real model's own forward machinery, and it comes
    for free now that the whole model is resident (no shard-streaming to conflict with running a
    real forward pass). It is ALSO what makes the sequential-GPTQ property automatic: `model` is
    the SAME object `pass_quantize_eval` is quantizing layer by layer, so by the time this runs for
    layer i, blocks 0..i-1 already hold their final (possibly quantized) weights, and this hook
    sees activations produced by them, not by their original float weights.

    `calibration_batches` should be an iterable of input_ids tensors of shape (1, seq_len).
    Returns H, a (d_in, d_in) float64 tensor.
    """
    d_in = module.weight.shape[0]           # Conv1D weight is (in_features, out_features)
    H = torch.zeros(d_in, d_in, dtype=torch.float64, device=device)
    n_samples = [0]

    def _hook(mod, inputs):
        x = inputs[0].detach()
        x = x.reshape(-1, x.shape[-1]).to(torch.float64)   # (tokens, d_in)
        H.add_(2.0 * x.T @ x)
        n_samples[0] += x.shape[0]

    handle = module.register_forward_pre_hook(_hook)
    try:
        model.eval()
        with torch.no_grad():
            for input_ids in calibration_batches:
                model(input_ids.to(device))
    finally:
        handle.remove()

    if n_samples[0] > 0:
        H /= n_samples[0]
    return H


def capture_c_attn_input(model, layer, batch, device):
    """
    One-shot capture of the tensor entering `layer.attn.c_attn` (i.e. ln_1(h)) from a REAL forward
    pass of `model` on `batch`, via a temporary forward pre-hook. The single-batch analogue of
    `collect_hessian_via_hook` above: used wherever the old code needed one specific batch's input
    activations for one specific layer (JAB-Hessian trace scoring in `pass_score`, and the STE
    fine-tuning targets in `pass_quantize_eval`) rather than an accumulated Hessian. Reflects the
    CURRENT state of every upstream block for the same reason `collect_hessian_via_hook` does: it's
    a real forward pass through the live, possibly-partially-quantized model.
    """
    captured = {}

    def _hook(mod, inputs):
        captured["X"] = inputs[0].detach().to(torch.float32)

    handle = layer.attn.c_attn.register_forward_pre_hook(_hook)
    try:
        with torch.no_grad():
            model(batch.to(device))
    finally:
        handle.remove()
    return captured["X"]


@torch.no_grad()
def _sliding_window_nll(model, ids, max_length, stride):
    """
    Sliding-window NLL sum and scored-token count over a single 1D token-id tensor. The shared
    core of `evaluate_perplexity` below (full WikiText-2 test) and section 7's tiny-model
    validation (synthetic ids) -- factored out so the tiny-model check exercises the SAME
    windowing/masking logic as the real evaluation without needing network access or a real
    tokenizer's vocabulary.
    """
    device = model.device
    ids = ids.to(device)
    seq_len = ids.shape[0]
    model.eval()
    nll_sum, n_tokens, prev_end = 0.0, 0, 0
    for begin in range(0, seq_len, stride):
        end = min(begin + max_length, seq_len)
        trg_len = end - prev_end
        input_ids = ids[begin:end].unsqueeze(0)
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100
        out = model(input_ids, labels=target_ids)
        nll_sum += out.loss.item() * trg_len
        n_tokens += trg_len
        prev_end = end
        if end == seq_len:
            break
    return nll_sum, n_tokens


@torch.no_grad()
def evaluate_perplexity(model, tokenizer, max_length=None, stride=None):
    """
    Sliding-window perplexity on the FULL WikiText-2 test set, using the model's OWN forward
    (`model(input_ids, labels=target_ids)`, i.e. HF's own cross-entropy) rather than a hand-rolled
    logit-chunking loop. Ported from the working reference notebook: every scored token gets
    `stride` tokens of real context (the standard sliding-window scheme), and scoring the full test
    set rather than a small window subset is what gives arm-to-arm differences enough resolution to
    be distinguishable from run-to-run noise.
    """
    max_length = EVAL_MAX_LENGTH if max_length is None else max_length
    stride = EVAL_STRIDE if stride is None else stride
    raw = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
    text = "\n\n".join(t for t in raw["text"] if t.strip())
    ids = tokenizer(text, return_tensors="pt").input_ids[0]
    nll_sum, n_tokens = _sliding_window_nll(model, ids, max_length, stride)
    return math.exp(nll_sum / n_tokens)

@torch.no_grad()
def _sliding_window_nll_acc(model, ids, max_length, stride):
    """Same windowing/masking as _sliding_window_nll above, plus next-token top-1 accuracy scored
    over the IDENTICAL positions (same shift, same -100 mask) in the same forward pass -- so
    perplexity and accuracy always cover exactly the same tokens."""
    device = model.device
    ids = ids.to(device)
    seq_len = ids.shape[0]
    model.eval()
    nll_sum, correct, n_tokens, prev_end = 0.0, 0, 0, 0
    for begin in range(0, seq_len, stride):
        end = min(begin + max_length, seq_len)
        trg_len = end - prev_end
        input_ids = ids[begin:end].unsqueeze(0)
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100
        out = model(input_ids, labels=target_ids)
        nll_sum += out.loss.item() * trg_len
        shift_logits = out.logits[:, :-1, :]
        shift_targets = target_ids[:, 1:]
        mask = shift_targets != -100
        preds = shift_logits.argmax(dim=-1)
        correct += ((preds == shift_targets) & mask).sum().item()
        n_tokens += trg_len
        prev_end = end
        if end == seq_len:
            break
    return nll_sum, correct, n_tokens


@torch.no_grad()
def evaluate_perplexity_and_accuracy(model, tokenizer, text, max_length=None, stride=None):
    """Perplexity + next-token top-1 accuracy over the SAME strided windows as evaluate_perplexity
    (identical windowing/masking, just also tracking argmax correctness), in ONE forward pass.
    `text` is a raw corpus string -- WikiText-2 test or a C4 slice -- unlike evaluate_perplexity,
    which always loads WikiText-2 itself; this is what lets the sweep score both eval datasets
    with one function."""
    max_length = EVAL_MAX_LENGTH if max_length is None else max_length
    stride = EVAL_STRIDE if stride is None else stride
    ids = tokenizer(text, return_tensors="pt").input_ids[0]
    nll_sum, correct, n_tokens = _sliding_window_nll_acc(model, ids, max_length, stride)
    return math.exp(nll_sum / n_tokens), correct / n_tokens


## 6. Calibration and evaluation data (C4)

Calibration switches from WikiText-2 to C4 (`allenai/c4`, `en`, streaming `train` split) -- the
standard GPTQ/AWQ protocol, feeding the Hessian, the JAB scoring pass, and the fine-tuning
batches alike. Evaluation text for both eval datasets (WikiText-2 test, C4 validation) is also
built here; `evaluate_perplexity`/`evaluate_perplexity_and_accuracy` (section 5) score whichever
text they're given with the same sliding-window logic.

In [9]:
from datasets import load_dataset


def build_calibration_ids(tokenizer, n_samples=None, seq_len=None, seed=None):
    """
    `n_samples` chunks of `seq_len` tokens from C4 (en, train, streaming) -- the standard
    GPTQ/AWQ calibration protocol, used instead of WikiText-2 so these numbers are comparable to
    published INT-quantization results. A document must tokenize to >= seq_len tokens to
    contribute a chunk; the chunk is a random contiguous span within it. `seed` makes the sample
    reproducible; the eval C4 slice (load_c4_eval_text, below) comes from the disjoint
    `validation` split, so there is no train/eval leakage to reason about.
    """
    n_samples = CALIB_N_SAMPLES if n_samples is None else n_samples
    seq_len = CALIB_SEQ_LEN if seq_len is None else seq_len
    seed = CALIB_SEED if seed is None else seed
    assert seq_len <= 1024, f"seq_len={seq_len} exceeds GPT-2's context length (1024)"

    rng = random.Random(seed)
    stream = load_dataset(C4_DATASET_NAME, C4_DATASET_CONFIG, split="train", streaming=True)
    stream = stream.shuffle(seed=seed, buffer_size=10_000)

    out = []
    for ex in stream:
        if len(out) >= n_samples:
            break
        ids = tokenizer(ex["text"], return_tensors="pt", add_special_tokens=False).input_ids[0]
        if ids.shape[0] < seq_len:
            continue
        start = rng.randint(0, ids.shape[0] - seq_len)
        out.append(ids[start:start + seq_len].unsqueeze(0))
    assert len(out) == n_samples, f"only found {len(out)}/{n_samples} C4 documents long enough"
    return out


def load_wikitext2_eval_text():
    """Full WikiText-2 test split, joined the same way evaluate_perplexity used to do internally."""
    raw = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
    return "\n\n".join(t for t in raw["text"] if t.strip())


def load_c4_eval_text(n_docs=None, seed=None):
    """
    C4 `validation` split, streamed and shuffled with a fixed seed, `n_docs` documents joined into
    one eval corpus. `validation` is a different split from calibration's `train` stream, so this
    is disjoint from the calibration sample by construction -- no manual offset bookkeeping.
    """
    n_docs = C4_EVAL_N_DOCS if n_docs is None else n_docs
    seed = CALIB_SEED if seed is None else seed
    stream = load_dataset(C4_DATASET_NAME, C4_DATASET_CONFIG, split="validation", streaming=True)
    stream = stream.shuffle(seed=seed, buffer_size=10_000)
    texts = [ex["text"] for ex, _ in zip(stream, range(n_docs))]
    return "\n\n".join(t for t in texts if t.strip())


def build_metric_eval_chunks(tokenizer, text, n_chunks, seq_len, seed):
    """`n_chunks` random contiguous `seq_len`-token spans from `text` -- the fixed-size batches
    the attention-reconstruction metric (section 8) averages over, built at CALIB_SEQ_LEN so they
    share pass_quantize_eval's causal mask."""
    rng = random.Random(seed)
    ids = tokenizer(text, return_tensors="pt").input_ids[0]
    out = []
    for _ in range(n_chunks):
        if ids.shape[0] <= seq_len:
            out.append(ids[:seq_len].unsqueeze(0))
            continue
        start = rng.randint(0, ids.shape[0] - seq_len)
        out.append(ids[start:start + seq_len].unsqueeze(0))
    return out


## 7. Validation

In [10]:
def _validate_gpt2_forward(verbose=True):
    """
    7a: hand-written block_forward vs. a real GPT2LMHeadModel, on a tiny random model.
    7b: _sliding_window_nll (the core of evaluate_perplexity) vs. HF's own labels=... loss, on
        the same tiny model and the same single window.
    Plus: the Conv1D orientation guard, and the STE-grid-is-a-no-op-on-GPTQ's-own-output check.
    """
    from transformers import GPT2Config, GPT2LMHeadModel

    torch.manual_seed(0)
    tiny = GPT2Config(vocab_size=256, n_embd=32, n_head=4, n_layer=3, n_positions=64,
                      bos_token_id=0, eos_token_id=0)
    ref = GPT2LMHeadModel(tiny)
    ref.config._attn_implementation = "eager"
    ref.eval()
    for p in ref.parameters():
        p.requires_grad_(False)

    cfg = AttnConfig(ref.config)
    assert (cfg.num_heads, cfg.num_kv_heads, cfg.head_dim, cfg.n_rep) == (4, 4, 8, 1), (
        "no GQA in GPT-2 -- num_kv_heads must equal num_heads and n_rep must be 1")
    assert cfg.q_out == cfg.kv_out == 32, "Q/K/V must be equal-sized thirds (no GQA shrink)"

    T = 24
    ids = torch.randint(0, 256, (1, T))
    with torch.no_grad():
        want = ref(ids).logits

    mask = build_attn_mask(T, "cpu")
    assert int(mask[T - 1].sum()) == T, "plain causal mask: the last row must attend to everything"

    # embed_cache reads DEVICE/GPU_DTYPE as globals (not parameters), so on a GPU runtime it would
    # move hidden states onto CUDA while `ref` -- deliberately never touched by .to(DEVICE) --
    # stays on CPU, and ln_1/ln_2/ln_f would mismatch devices. This tiny check is meant to run in
    # seconds regardless of accelerator, so force it onto CPU/fp32 for its duration and restore
    # afterward, the same guard `_validate_adaptive_joint_pass` already uses.
    old_device, old_dtype = globals()["DEVICE"], globals()["GPU_DTYPE"]
    globals()["DEVICE"], globals()["GPU_DTYPE"] = "cpu", torch.float32
    try:
        with torch.no_grad():
            h = embed_cache(ref, [ids])[0]
            for i in range(cfg.n_layers):
                h, attn_w = block_forward(ref.transformer.h[i], h, cfg, mask, return_attn=True)
                assert attn_w.shape == (1, cfg.num_heads, T, T)
            got = ref.lm_head(ref.transformer.ln_f(h))

        rel = ((got - want).norm() / want.norm()).item()
        if verbose:
            print(f"  7a: hand-written forward vs. GPT2LMHeadModel.forward: relative error "
                  f"{rel:.3e}")
        assert rel < 1e-3, (
            f"hand-written forward disagrees with the real model (rel err {rel:.3e}). Check the "
            f"Conv1D orientation, the c_attn Q/K/V split order, and the causal mask.")

        # Prove the check is not vacuous: disable the causal mask and confirm attention moves.
        with torch.no_grad():
            h0 = embed_cache(ref, [ids])[0]
            _, attn_ok = block_forward(ref.transformer.h[0], h0, cfg, mask, return_attn=True)
            full_mask = torch.ones(T, T, dtype=torch.bool)
            _, attn_no = block_forward(ref.transformer.h[0], h0, cfg, full_mask, return_attn=True)
        rel_broken = ((attn_no - attn_ok).norm() / attn_ok.norm()).item()
        assert rel_broken > 1e-2, "disabling the causal mask changed nothing -- this check is vacuous!"
        if verbose:
            print(f"  causal mask is load-bearing (disabling it moves attention maps by "
                  f"{rel_broken:.3e})")

        # 7b: _sliding_window_nll (section 5 -- the shared core of evaluate_perplexity) against
        # HF's own labels=... loss, on the SAME single window (stride == max_length == T).
        nll_ours, n_ours = _sliding_window_nll(ref, ids[0], max_length=T, stride=T)
        ppl_ours = math.exp(nll_ours / n_ours)
        with torch.no_grad():
            ppl_hf = math.exp(ref(ids, labels=ids).loss.item())
        rel_ppl = abs(ppl_ours - ppl_hf) / ppl_hf
        if verbose:
            print(f"  7b: _sliding_window_nll vs. HF's labels=... loss: ours={ppl_ours:.4f} "
                  f"hf={ppl_hf:.4f} (rel err {rel_ppl:.3e})")
        assert rel_ppl < 1e-3, (
            f"_sliding_window_nll disagrees with HF's own loss (rel err {rel_ppl:.3e})")
    finally:
        globals()["DEVICE"], globals()["GPU_DTYPE"] = old_device, old_dtype

    # The STE grid must be a no-op on GPTQ's own output (the section-13 warm-start guarantee).
    # CPU-only, no model involved -- no device guard needed.
    W0 = torch.randn(32, 32) * 0.05
    Xd = torch.randn(2000, 32)
    Hd = (2.0 * Xd.T @ Xd / Xd.shape[0]).double()
    Wq, sc = gptq_quantize_layer(W0.clone(), Hd, bits=4, group_size=8, act_order=True,
                                 return_scale=True, work_dtype=torch.float64)
    qmax = 7
    rel_ste = ((torch.clamp(torch.round(Wq / sc), -qmax, qmax) * sc - Wq).norm() / Wq.norm()).item()
    assert rel_ste < 1e-6, f"return_scale is not the grid GPTQ used (rel err {rel_ste:.3e})"
    if verbose:
        print(f"  GPTQ return_scale is exact (re-quantization relative error {rel_ste:.3e})")

    # Explicit Conv1D orientation guard: quantize a (d_in, d_out) module and confirm the shape
    # survives and x @ W still type-checks -- this is the check for the bug class described in
    # gptq_quantize_conv1d's docstring (section 1).
    class _FakeConv1D:
        def __init__(self, w):
            self.weight = torch.nn.Parameter(w.clone())

    d_in, d_out = 16, 48
    fake = _FakeConv1D(torch.randn(d_in, d_out) * 0.05)
    Xc = torch.randn(500, d_in)
    Hc = (2.0 * Xc.T @ Xc / Xc.shape[0]).double()
    W_before_shape = tuple(fake.weight.data.shape)
    W_after = gptq_quantize_conv1d(fake, Hc, bits=4, group_size=8)
    assert tuple(W_after.shape) == W_before_shape == (d_in, d_out), (
        f"gptq_quantize_conv1d changed the weight orientation: {W_before_shape} -> "
        f"{tuple(W_after.shape)}")
    _ = Xc @ fake.weight.data     # must not raise a shape error
    if verbose:
        print(f"  gptq_quantize_conv1d preserves the (d_in, d_out) Conv1D orientation "
              f"{tuple(fake.weight.data.shape)}, and x @ W still type-checks")

    free(ref)
    return rel


print("Validation 7a/7b: hand-written forward + perplexity path vs. real GPT2LMHeadModel...")
_validate_gpt2_forward()
print("PASSED -- the hand-written forward and the perplexity path reproduce GPT2LMHeadModel.\n")


def _validate_adaptive_joint_pass(verbose=True):
    """
    7c: pass_quantize_eval(bits_fn=..., finetune=True) on a hand-made mixed 2/3/4/8-bit
    assignment, tiny random GPT-2, CPU-only, seconds not minutes. Exercises exactly the
    section-13b code path (adaptive allocation + joint fine-tuning together) that sections 7a/7b,
    11, 12 and 13 each individually miss.

    `reader` in this notebook IS the resident model, so a tiny in-memory `GPT2LMHeadModel` can be
    passed directly to pass_quantize_eval -- no CheckpointReader-shaped stand-in needed. The one
    thing that DOES need a stand-in is `eval_fn`: the real pipeline's eval_fn closes over
    evaluate_perplexity, which downloads real WikiText-2 -- this test instead builds a synthetic
    "test set" from random ids and scores it with the SAME _sliding_window_nll evaluate_perplexity
    itself uses, keeping this test network-free and CPU-only.
    """
    from transformers import GPT2Config, GPT2LMHeadModel

    torch.manual_seed(1)
    tiny = GPT2Config(vocab_size=256, n_embd=32, n_head=4, n_layer=4, n_positions=64,
                      bos_token_id=0, eos_token_id=0)
    ref = GPT2LMHeadModel(tiny)
    ref.config._attn_implementation = "eager"
    ref.eval()
    for p in ref.parameters():
        p.requires_grad_(False)

    cfg = AttnConfig(ref.config)

    # every bit-width this notebook supports except 16 (kept out only for speed); 2-bit is
    # included deliberately -- the allocator can and does emit it, and it must not be silently
    # clamped up to some higher floor.
    mixed_bits = {0: 2, 1: 3, 2: 4, 3: 8}
    assert set(mixed_bits) == set(range(cfg.n_layers)), "mixed_bits must cover every tiny layer"

    calib_ids = [torch.randint(0, 256, (1, 16)) for _ in range(4)]
    eval_ids = torch.randint(0, 256, (48,))          # synthetic "test set" stand-in

    def synthetic_eval_fn():
        nll, n = _sliding_window_nll(ref, eval_ids, max_length=16, stride=16)
        return math.exp(nll / n)

    # pass_quantize_eval reads DEVICE/GPU_DTYPE as globals, not as parameters -- swap them to
    # CPU/fp32 for this check, then restore.
    old_device, old_dtype = globals()["DEVICE"], globals()["GPU_DTYPE"]
    globals()["DEVICE"], globals()["GPU_DTYPE"] = "cpu", torch.float32
    try:
        ppl, notes = pass_quantize_eval(
            ref, ref.config, cfg, calib_ids, synthetic_eval_fn,
            bits_fn=lambda i: mixed_bits[i], finetune=True,
            steps_per_block=2, n_hessian_batches=2, group_size=8,
            label="7c: mixed-bit adaptive+joint (tiny CPU model)", verbose=False,
            return_notes=True)
    finally:
        globals()["DEVICE"], globals()["GPU_DTYPE"] = old_device, old_dtype

    assert math.isfinite(ppl) and ppl > 0, f"non-finite/degenerate perplexity: {ppl}"
    assert set(notes) == set(mixed_bits), (
        f"notes missing layers: {set(mixed_bits) - set(notes)} -- return_notes must record "
        f"every bits_fn'd layer, regardless of bit-width")
    for i, bits in mixed_bits.items():
        got_bits, committed = notes[i]
        assert got_bits == bits, f"layer {i}: note recorded {got_bits} bits, assigned {bits}"
        assert committed in (True, False), (
            f"layer {i} ({bits}-bit): finetune=True but committed={committed!r} -- want_scales "
            f"must not be gated on bit-width, so even the 2-bit layer needs a real fine-tune "
            f"attempt and a True/False outcome, not None")

    if verbose:
        detail = ", ".join(f"L{i}={b}b/{'FT' if c else 'GPTQ'}"
                           for i, (b, c) in sorted(notes.items()))
        print(f"  mixed 2/3/4/8-bit adaptive+joint pass ran end-to-end on a tiny CPU GPT-2 model "
              f"(ppl={ppl:.3f}): {detail}")

    free(ref)
    return ppl

Validation 7a/7b: hand-written forward + perplexity path vs. real GPT2LMHeadModel...
  7a: hand-written forward vs. GPT2LMHeadModel.forward: relative error 0.000e+00


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


  causal mask is load-bearing (disabling it moves attention maps by 8.576e-01)
  7b: _sliding_window_nll vs. HF's labels=... loss: ours=255.8802 hf=255.8802 (rel err 0.000e+00)
  GPTQ return_scale is exact (re-quantization relative error 2.487e-08)
  gptq_quantize_conv1d preserves the (d_in, d_out) Conv1D orientation (16, 48), and x @ W still type-checks
PASSED -- the hand-written forward and the perplexity path reproduce GPT2LMHeadModel.



## 8. The pipeline passes

In [11]:
class STEQuantize(torch.autograd.Function):
    """Straight-through on w; LSQ gradient on scale (Esser et al.), scaled by 1/sqrt(n_weights*qmax)
    since each group's scale now accumulates gradient from n_weights shared weights."""

    @staticmethod
    def forward(ctx, w, scale, bits, n_weights):
        qmax = 2 ** (bits - 1) - 1
        v = w / scale
        q = torch.clamp(torch.round(v), -qmax, qmax)
        ctx.save_for_backward(v, q)
        ctx.qmax = qmax
        ctx.grad_scale_factor = (n_weights * qmax) ** -0.5
        return q * scale

    @staticmethod
    def backward(ctx, grad_output):
        v, q = ctx.saved_tensors
        qmax = ctx.qmax
        grad_scale_local = torch.where(v.abs() <= qmax, q - v, torch.sign(v) * qmax)
        return grad_output, grad_output * grad_scale_local * ctx.grad_scale_factor, None, None


def ste_quantize(w, scale, bits, n_weights):
    return STEQuantize.apply(w, scale, bits, n_weights)


def expand_scale(scale_compact, g_idx, cfg):
    """(3*hidden, n_groups) scale, one per (output row, group), shared g_idx (d_in,) -> a
    per-weight tensor flattened in the same [Q|K|V] order as flatten_weights/w_flat."""
    hs = cfg.hidden_size
    parts = [chunk[:, g_idx].T.reshape(-1)
            for chunk in (scale_compact[:hs], scale_compact[hs:2 * hs], scale_compact[2 * hs:])]
    return torch.cat(parts)


def pass_score(reader, config, cfg, calib_ids, *, samples=None, n_score_batches=None,
               n_hessian_batches=None, lambda_kl=None, group_size=None, bit_widths=None,
               perturb_mode=None, verbose=True):
    """
    Scoring pass over an all-float model. `reader` MUST be a freshly-loaded, unquantized model --
    every caller in section 11 gets one from `load_model_and_tokenizer` for exactly this reason
    (see that function's docstring, section 5): scoring against an already-quantized model would
    measure "how sensitive is the already-degraded model", not "where should bits go", defeating
    the point of adaptive allocation.

    Returns (jab_scores, allocator_input):
      jab_scores      -- {layer_name: Hessian trace of the attention-aware loss}
      allocator_input -- HAWQ-V2 style {layer_name: {bits: trace * ||Q(W)-W||_F^2}}
    """
    samples = HUTCH_SAMPLES if samples is None else samples
    n_score_batches = JAB_N_BATCHES if n_score_batches is None else n_score_batches
    n_hessian_batches = HESSIAN_N_BATCHES if n_hessian_batches is None else n_hessian_batches
    lambda_kl = LAMBDA_KL if lambda_kl is None else lambda_kl
    group_size = GROUP_SIZE if group_size is None else group_size
    bit_widths = BIT_WIDTHS if bit_widths is None else bit_widths
    perturb_mode = PERTURB_MODE if perturb_mode is None else perturb_mode

    reset_vram_peak()
    mask = build_attn_mask(calib_ids[0].shape[1], DEVICE)

    jab_scores, allocator_input = {}, {}
    print(f"Scoring {cfg.n_layers} layers: {samples} Hutchinson probes x {n_score_batches} "
          f"batches, loss = MSE + {lambda_kl}*KL, perturbation mode '{perturb_mode}'")

    for i in range(cfg.n_layers):
        layer = load_decoder_layer(reader, config, i)
        name = layer_name(i)

        # --- Hessian H = 2 X^T X (shared by q/k/v -- one fused matrix, one Hessian), via a real
        # forward pre-hook on this layer's c_attn (section 5) rather than a hand-propagated cache.
        H = collect_hessian_via_hook(reader, layer.attn.c_attn, calib_ids[:n_hessian_batches],
                                     DEVICE)

        # --- JAB-Hessian trace of the attention-aware loss ---
        W32 = qkv_weights_io(layer, dtype=torch.float32)
        b32 = qkv_bias_io(layer, dtype=torch.float32)
        w_flat = flatten_weights(*W32).clone().requires_grad_(True)
        traces = []
        for b in calib_ids[:n_score_batches]:
            # X: this layer's REAL input, from a real forward pass of `reader` on batch `b`
            # (section 5's capture_c_attn_input) -- not a hand-computed ln_1(h).
            X = capture_c_attn_input(reader, layer, b, DEVICE)
            with torch.no_grad():
                # Targets come from this layer's own float weights, so the loss is exactly 0 at
                # w = w_true and the trace is the curvature of the reconstruction loss there.
                tA, tattn = compute_attention(*W32, X, cfg, b_Q=b32[0], b_K=b32[1], b_V=b32[2],
                                              attn_mask=mask)
            tattn = tattn if lambda_kl else None

            def loss_fn(p):
                return attention_loss(p, X, tA, cfg, target_attn=tattn, lambda_kl=lambda_kl,
                                      b_Q=b32[0], b_K=b32[1], b_V=b32[2], attn_mask=mask)

            traces.append(hutchinson_trace_estimator(loss_fn, w_flat, samples=samples).item())
            free(X, tA, tattn)
        trace = sum(traces) / len(traces)
        jab_scores[name] = trace
        free(w_flat)

        # --- HAWQ-V2 table: Omega_i(bits) = trace_i * ||Q(W_i) - W_i||_F^2, summed over q/k/v ---
        table = {}
        for bits in bit_widths:
            pert = sum(grid_perturbation(W, bits, group_size=group_size, H=H, mode=perturb_mode)
                       for W in W32)
            table[bits] = trace * pert
        allocator_input[name] = table
        del H, W32                    # del, not free(...) -- see free()'s docstring
        free()

        if verbose:
            level = "HIGH" if trace > 100 else "MEDIUM" if trace > 10 else "LOW"
            print(f"  {name}: trace={trace:.4f} [{level}]  " +
                  ", ".join(f"{b}b={v:.3e}" for b, v in table.items()))

        del layer
        free()

    free(mask)
    vram("after scoring pass")
    return jab_scores, allocator_input


def pass_quantize_eval(reader, config, cfg, calib_ids, eval_fn, *, bits_fn=None,
                       finetune=False, n_hessian_batches=None, group_size=None, lambda_kl=None,
                       steps_per_block=None, lr=None, grad_clip_norm=None, layer_indices=None,
                       label="", verbose=True, return_notes=False,
                       collect_metrics=False, metric_eval_chunks=None):
    """
    Quantize-and-evaluate pass. `reader` MUST be a freshly-loaded model whenever `bits_fn` is not
    None -- quantization mutates `c_attn.weight` in place, so an already-quantized `reader` would
    have this arm quantize an already-quantized matrix (see load_model_and_tokenizer's docstring).

    eval_fn: zero-argument callable returning the perplexity to report once every layer has been
        processed, e.g. `lambda: evaluate_perplexity(reader, tokenizer, ...)`. Factored out (rather
        than calling evaluate_perplexity directly in here) purely so section 7's tiny-model
        validation can inject a synthetic evaluator that needs no network access -- the real
        pipeline always passes a closure over evaluate_perplexity.

    bits_fn: None -> leave weights untouched (the control); else layer_idx -> bit-width.
    finetune: additionally run the STE joint fine-tune of Objective 1 on each layer, against a
        float-weight teacher applied to the SAME input the student sees (GPTQ's own convention:
        target = W_float @ X_quant -- there is no separate float hidden-state trajectory here).
    layer_indices: restrict fine-tuning to these layers; the rest still get the GPTQ warm start.
    return_notes: if True, also return {layer_idx: (bits, committed)} where `committed` is True
        if a fine-tuned weight beat the GPTQ-only starting point and was written back, False if
        GPTQ-only was kept, None if the layer had no bits_fn/finetune outcome to record.
    collect_metrics: if True, ALSO return a dict {"quant_error": {layer_idx: relative Frobenius
        error on c_attn's weight}, "attn_recon": {dataset_name: {layer_idx: mean relative
        Frobenius error on the merged pre-c_proj attention output}}} -- the latter averaged over
        `metric_eval_chunks[dataset_name]`, teacher = the float weights snapshotted just before
        this layer is quantized, student = the layer's current (quantized, possibly fine-tuned)
        weights, both run on the SAME input captured from `reader` (so, from the live, partially
        quantized model, same convention as the FT target above). Both metrics are skipped for
        layers where bits_fn is None (nothing was quantized).

    Returns perplexity, or a tuple adding notes and/or metrics in that order depending on which
    of return_notes/collect_metrics is set.
    """
    n_hessian_batches = HESSIAN_N_BATCHES if n_hessian_batches is None else n_hessian_batches
    group_size = GROUP_SIZE if group_size is None else group_size
    lambda_kl = LAMBDA_KL if lambda_kl is None else lambda_kl
    steps_per_block = JOINT_STEPS_PER_BLOCK if steps_per_block is None else steps_per_block
    lr = JOINT_LR if lr is None else lr
    grad_clip_norm = JOINT_GRAD_CLIP if grad_clip_norm is None else grad_clip_norm
    tune = None if layer_indices is None else set(layer_indices)

    print(f"\n=== pass: {label} ===")
    reset_vram_peak()
    mask = build_attn_mask(calib_ids[0].shape[1], DEVICE)

    notes = {}
    flip_rates = {}
    quant_errors = {}
    attn_recon = {ds: {} for ds in (metric_eval_chunks or {})} if collect_metrics else {}
    for i in range(cfg.n_layers):
        layer = load_decoder_layer(reader, config, i)
        note = ""
        note_committed = None

        # (1) float snapshot, BEFORE the module is quantized. The bias is fixed (never a
        # GPTQ/STE target) but still snapshotted alongside the weight, since both feed the
        # teacher's compute_attention call below (for fine-tuning AND for the attn-recon metric).
        W_float32 = None
        need_float_snapshot = finetune or collect_metrics
        if need_float_snapshot:
            W_float32 = tuple(W.clone() for W in qkv_weights_io(layer, dtype=torch.float32))
            b_Q, b_K, b_V = qkv_bias_io(layer, dtype=torch.float32)
        W_c_attn_float = (layer.attn.c_attn.weight.data.clone().to(torch.float32)
                          if collect_metrics else None)

        if bits_fn is not None:
            bits = bits_fn(i)
            # (2) GPTQ warm start. `reader` is quantized layer by layer in place, and
            # collect_hessian_via_hook runs a REAL forward pass through the current (possibly
            # partially-quantized) `reader` -- so the Hessian for layer i automatically reflects
            # layers 0..i-1 already being quantized, exactly the sequential-GPTQ behaviour.
            H = collect_hessian_via_hook(reader, layer.attn.c_attn,
                                         calib_ids[:n_hessian_batches], DEVICE)
            want_scales = finetune and (tune is None or i in tune)
            out = quantize_qkv(qkv_of(layer), H, bits, group_size=group_size,
                               return_scales=want_scales, return_compact=want_scales)
            free(H)
            note = f"{bits} bits"

            # (3) STE joint fine-tune against the float reference
            if want_scales:
                ws, scales, scale_compact, g_idx = out
                # w_flat is seeded from GPTQ's fp32 output (not the fp16 stored weight) and
                # scale_flat is GPTQ's own grid, so ste_quantize is a no-op at step 0.
                w_flat = flatten_weights(*ws).clone().requires_grad_(True)
                gptq_w = w_flat.detach().clone()          # diagnostic B: pre-fine-tune grid point
                scale_flat = flatten_weights(*scales).to(w_flat.dtype)   # frozen GPTQ grid
                # one learnable scale per (output row, group) -- (3*hidden, n_groups), NOT one per
                # weight: a per-weight scale makes q_i*s_i free precision, not bits-bit quantization
                scale_param = scale_compact.clone().requires_grad_(True)
                n_weights = group_size                    # weights sharing each group's scale
                free(ws, scales)
                lr_w = ALPHA_W * scale_flat.mean().item()
                lr_s = ALPHA_S * scale_flat.mean().item()
                opt = torch.optim.Adam([{"params": [w_flat], "lr": lr_w},
                                        {"params": [scale_param], "lr": lr_s}])
                sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps_per_block)

                # held_out is reserved out of the cycling pool entirely (not just the next index
                # after `order`) so it stays held out even when steps_per_block > len(calib_ids)-1
                # and `order` wraps around the whole pool via modulo.
                held_out = len(calib_ids) - 1
                pool = list(range(len(calib_ids) - 1))
                start = (i * steps_per_block) % len(pool)
                order = [pool[(start + s) % len(pool)] for s in range(steps_per_block)]

                def targets(j):
                    # Teacher and student see the SAME input (GPTQ's own convention: target =
                    # W_float @ X_quant). A separate float-hidden-state trajectory would let the
                    # residual include upstream quantization error this layer cannot fix. X comes
                    # from a real forward pass of `reader` (capture_c_attn_input, section 5), so it
                    # reflects layers 0..i-1's CURRENT (already-quantized) state, same as (2) above.
                    X = capture_c_attn_input(reader, layer, calib_ids[j], DEVICE)
                    with torch.no_grad():
                        tA, tattn = compute_attention(*W_float32, X, cfg, b_Q=b_Q, b_K=b_K,
                                                      b_V=b_V, attn_mask=mask)
                    return X, tA, (tattn if lambda_kl else None)

                # X depends only on layers 0..i-1, frozen for layer i's whole FT -- compute each
                # distinct batch's targets once (capture_c_attn_input is a full model forward) and
                # reuse them instead of recomputing every step.
                batch_cache = {j: targets(j) for j in set(order) | {held_out}}

                def score(w, s):
                    with torch.no_grad():
                        X_ho, tA_ho, tattn_ho = batch_cache[held_out]
                        return attention_loss(
                            ste_quantize(w, expand_scale(s, g_idx, cfg), bits, n_weights),
                            X_ho, tA_ho, cfg, target_attn=tattn_ho, lambda_kl=lambda_kl,
                            b_Q=b_Q, b_K=b_K, b_V=b_V, attn_mask=mask).item()

                init_loss = score(w_flat.detach(), scale_param.detach())
                best_loss, best_w = init_loss, w_flat.detach().clone()
                for step_idx, j in enumerate(order):
                    X, tA, tattn = batch_cache[j]
                    opt.zero_grad()
                    scale_expanded = expand_scale(scale_param, g_idx, cfg)
                    loss = attention_loss(ste_quantize(w_flat, scale_expanded, bits, n_weights),
                                          X, tA, cfg, target_attn=tattn, lambda_kl=lambda_kl,
                                          b_Q=b_Q, b_K=b_K, b_V=b_V, attn_mask=mask,
                                          log_components=LOG_LOSS_COMPONENTS)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_([w_flat, scale_param], grad_clip_norm)
                    opt.step()
                    sched.step()
                    # score every SCORE_EVERY steps (always on the last) instead of every step --
                    # held-out targets are cached above, so this is just the loss/quantize forward.
                    if step_idx == steps_per_block - 1 or (step_idx + 1) % SCORE_EVERY == 0:
                        step_loss = score(w_flat.detach(), scale_param.detach())
                        if step_loss < best_loss:
                            best_loss = step_loss
                            with torch.no_grad():
                                best_w = ste_quantize(w_flat, expand_scale(scale_param, g_idx, cfg),
                                                      bits, n_weights).detach().clone()
                    free(loss)
                del batch_cache
                free()

                # diagnostic B: fraction of grid positions fine-tuning actually moved, regardless
                # of whether the result was committed -- against the FROZEN GPTQ grid (scale_flat)
                with torch.no_grad():
                    flip_rate = (torch.round(gptq_w / scale_flat) !=
                                torch.round(best_w / scale_flat)).float().mean().item()

                # Safety net: commit only if fine-tuning beat the GPTQ-only starting point.
                # Otherwise the module already holds that GPTQ-only solution -- leave it alone.
                if best_loss <= init_loss:
                    write_qkv_flat(layer, best_w, cfg)
                    note += f", FT {init_loss:.6f} -> {best_loss:.6f}, flip rate {flip_rate:.3f}"
                    note_committed = True
                else:
                    note += (f", kept GPTQ-only ({init_loss:.6f} vs FT {best_loss:.6f}), "
                            f"flip rate {flip_rate:.3f}")
                    note_committed = False
                flip_rates[i] = flip_rate
                del w_flat, gptq_w, scale_flat, scale_param, best_w, opt, sched, targets, score
                free()
            elif finetune:
                note += ", GPTQ-only (not in layer_indices)"

            # (4) metrics 3 & 4 -- AFTER quantization/FT, so the module holds its final weights.
            if collect_metrics:
                W_q_full = layer.attn.c_attn.weight.data.to(torch.float32)
                quant_errors[i] = ((W_q_full - W_c_attn_float).norm()
                                   / W_c_attn_float.norm()).item()

                if metric_eval_chunks:
                    W_q32 = qkv_weights_io(layer, dtype=torch.float32)
                    for ds_name, chunks in metric_eval_chunks.items():
                        errs = []
                        for ch in chunks:
                            X = capture_c_attn_input(reader, layer, ch, DEVICE)
                            with torch.no_grad():
                                A_teacher, _ = compute_attention(*W_float32, X, cfg, b_Q=b_Q,
                                                                 b_K=b_K, b_V=b_V, attn_mask=mask)
                                A_student, _ = compute_attention(*W_q32, X, cfg, b_Q=b_Q, b_K=b_K,
                                                                 b_V=b_V, attn_mask=mask)
                            errs.append(((A_student - A_teacher).norm()
                                        / A_teacher.norm()).item())
                            free(X, A_teacher, A_student)
                        attn_recon[ds_name][i] = sum(errs) / len(errs)
                    del W_q32
                del W_q_full
        del W_float32, W_c_attn_float
        free()

        del layer
        free()

        if bits_fn is not None:
            notes[i] = (bits, note_committed)
        if verbose:
            print(f"  layer {i:>2}/{cfg.n_layers}: {note or 'fp32 (no quantization)'}")

    free(mask)
    globals()["LAST_FLIP_RATES"] = flip_rates
    ppl = eval_fn()
    vram(f"after '{label}'")
    print(f"=== {label}: perplexity {ppl:.3f} ===")

    if collect_metrics:
        metrics = {"quant_error": quant_errors, "attn_recon": attn_recon}
        return (ppl, notes, metrics) if return_notes else (ppl, metrics)
    return (ppl, notes) if return_notes else ppl


print("Validation 7c (deferred from section 7 -- needs pass_quantize_eval, defined just above): "
      "adaptive allocation + joint fine-tuning, mixed 2/3/4/8-bit (tiny CPU model)...")
_validate_adaptive_joint_pass()
print("PASSED -- pass_quantize_eval(bits_fn=..., finetune=True) is bit-width-agnostic end to "
      "end.")


Validation 7c (deferred from section 7 -- needs pass_quantize_eval, defined just above): adaptive allocation + joint fine-tuning, mixed 2/3/4/8-bit (tiny CPU model)...

=== pass: 7c: mixed-bit adaptive+joint (tiny CPU model) ===
    [vram after '7c: mixed-bit adaptive+joint (tiny CPU model)': 0.00 GB now, 0.00 GB peak this pass]
=== 7c: mixed-bit adaptive+joint (tiny CPU model): perplexity 257.434 ===
  mixed 2/3/4/8-bit adaptive+joint pass ran end-to-end on a tiny CPU GPT-2 model (ppl=257.434): L0=2b/FT, L1=3b/FT, L2=4b/FT, L3=8b/FT
PASSED -- pass_quantize_eval(bits_fn=..., finetune=True) is bit-width-agnostic end to end.


## 9. Setup

In [12]:
print(f"Loading {MODEL_ID}'s tokenizer and config (cached after the first run)...")
# Only the tokenizer and config are loaded here, NOT a model -- every arm in section 12 loads its
# OWN fresh model (see load_model_and_tokenizer's docstring, section 5) right before it quantizes
# anything, so there is no shared, mutable `reader` for one arm's damage to leak into the next.
tokenizer = GPT2TokenizerFast.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
config = AutoConfig.from_pretrained(MODEL_ID)
config._attn_implementation = "eager"

ATTN = AttnConfig(config)
print(ATTN)
assert ATTN.hidden_size % GROUP_SIZE == 0, (
    f"GROUP_SIZE={GROUP_SIZE} must divide hidden_size={ATTN.hidden_size} -- true for gpt2 (768), "
    f"gpt2-medium (1024) and gpt2-large (1280), but NOT gpt2-xl (1600)")
print(f"Q/K/V parameters per layer: {ATTN.qkv_numel:,}")

print(f"\nBuilding the C4 calibration set (train split, streaming, seed {CALIB_SEED})...")
calib_ids = build_calibration_ids(tokenizer)
print(f"{len(calib_ids)} calibration chunks of {CALIB_SEQ_LEN} tokens")

print(f"\nBuilding eval corpora: WikiText-2 test + C4 validation ({C4_EVAL_N_DOCS} docs, "
      "disjoint from calibration by split)...")
wikitext2_eval_text = load_wikitext2_eval_text()
c4_eval_text = load_c4_eval_text()
EVAL_TEXTS = {"wikitext2": wikitext2_eval_text, "c4": c4_eval_text}
print(f"Evaluation: max_length={EVAL_MAX_LENGTH}, stride={EVAL_STRIDE}, "
      f"scored on both {list(EVAL_TEXTS)}.")

print(f"\nBuilding {ATTN_RECON_N_CHUNKS} attention-reconstruction eval chunks per dataset...")
metric_eval_chunks = {
    "wikitext2": build_metric_eval_chunks(tokenizer, wikitext2_eval_text, ATTN_RECON_N_CHUNKS,
                                          CALIB_SEQ_LEN, seed=CALIB_SEED + 1),
    "c4": build_metric_eval_chunks(tokenizer, c4_eval_text, ATTN_RECON_N_CHUNKS,
                                   CALIB_SEQ_LEN, seed=CALIB_SEED + 2),
}

results_rows = []       # long-form: one dict per (mode, avg_bits, eval_dataset)
per_layer_store = {}    # {f"{mode}_{avg_bits}": {"quant_error": {...}, "attn_recon": {ds: {...}}}}


def eval_all(reader):
    """Perplexity + top-1 accuracy on both eval datasets, one forward pass each."""
    return {ds: evaluate_perplexity_and_accuracy(reader, tokenizer, text,
                                                 max_length=EVAL_MAX_LENGTH, stride=EVAL_STRIDE)
           for ds, text in EVAL_TEXTS.items()}


Loading gpt2's tokenizer and config (cached after the first run)...


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Evaluation: max_length=1024, stride=512, scored on both ['wikitext2', 'c4'].

Building 8 attention-reconstruction eval chunks per dataset...


## 10. fp32 control

In [13]:
reader, _ = load_model_and_tokenizer(MODEL_ID, DEVICE)   # fresh, unquantized control
for ds_name, (ppl, acc) in eval_all(reader).items():
    results_rows.append({"mode": "fp32", "avg_bits": None, "eval_dataset": ds_name,
                         "perplexity": ppl, "accuracy": acc,
                         "quant_error_mean": 0.0, "attn_recon_error_mean": 0.0})
    print(f"fp32 control [{ds_name}]: perplexity {ppl:.3f}, accuracy {acc:.4f}")
    if not (15.0 < ppl < 60.0):
        print(f"  WARNING: perplexity {ppl:.3f} is outside the expected ~18-30 range for "
              f"{MODEL_ID} -- investigate the pipeline before reading anything into the sweep.")


model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

fp32 control [wikitext2]: perplexity 24.357, accuracy 0.4148
fp32 control [c4]: perplexity 32.250, accuracy 0.3779


## 11. JAB-Hessian scoring (one pass, reused by every budget)

In [14]:
reader, _ = load_model_and_tokenizer(MODEL_ID, DEVICE)   # fresh, unquantized -- see pass_score's
                                                          # docstring for why scoring needs this
torch.manual_seed(42)
jab_scores, allocator_input = pass_score(reader, config, ATTN, calib_ids)


def jab_assignment_for(avg_bits):
    """Greedy allocation at a given average-bits budget -- reuses `allocator_input` from the ONE
    scoring pass above; scores don't depend on the budget, so this just re-solves the greedy
    knapsack, not a re-scoring."""
    budget = avg_bits * len(allocator_input)
    assignment, cost_used, sensitivity = greedy_allocate(allocator_input, budget)
    print(f"  avg_bits={avg_bits}: budget {budget:.1f}, used {cost_used:.1f} "
          f"({cost_used / len(allocator_input):.2f} avg), sensitivity {sensitivity:.4e}")
    return assignment


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Scoring 12 layers: 10 Hutchinson probes x 2 batches, loss = MSE + 0.1*KL, perturbation mode 'rtn'
  layer_0_QKV: trace=6625.5840 [HIGH]  2b=2.855e+08, 3b=4.586e+07, 4b=8.580e+06, 8b=2.612e+04, 16b=3.925e-01
  layer_1_QKV: trace=18708.2812 [HIGH]  2b=3.876e+08, 3b=5.396e+07, 4b=1.016e+07, 8b=3.103e+04, 16b=4.668e-01
  layer_2_QKV: trace=108362.1680 [HIGH]  2b=2.672e+09, 3b=3.888e+08, 4b=7.313e+07, 8b=2.235e+05, 16b=3.357e+00
  layer_3_QKV: trace=286301.2656 [HIGH]  2b=6.099e+09, 3b=8.652e+08, 4b=1.618e+08, 8b=4.921e+05, 16b=7.391e+00
  layer_4_QKV: trace=134667.0469 [HIGH]  2b=3.052e+09, 3b=4.437e+08, 4b=8.412e+07, 8b=2.552e+05, 16b=3.840e+00
  layer_5_QKV: trace=130668.1797 [HIGH]  2b=2.241e+09, 3b=2.999e+08, 4b=5.561e+07, 8b=1.693e+05, 16b=2.547e+00
  layer_6_QKV: trace=117275.3125 [HIGH]  2b=1.978e+09, 3b=2.664e+08, 4b=4.968e+07, 8b=1.513e+05, 16b=2.268e+00
  layer_7_QKV: trace=101614.9102 [HIGH]  2b=1.772e+09, 3b=2.363e+08, 4b=4.395e+07, 8b=1.339e+05, 16b=2.008e+00
  layer_8_QKV: tr

## 12. Mode x bit-budget sweep

`uniform`/`joint` apply one flat bit-width to every layer (undefined at fractional
`SWEEP_BITS` -- see `UNIFORM_AVAILABLE_WIDTHS`, section 2). `jab`/`adaptive_joint`/`kl0` mix
per-layer bit-widths via `jab_assignment_for` to hit any average budget, so all five
`SWEEP_BITS` entries are defined for them.

In [15]:
def run_arm(mode, avg_bits):
    """
    Fresh model load, quantize + evaluate one (mode, avg_bits) arm on both eval datasets.
    Returns (rows, per_layer_metrics), or None if this arm is undefined (a flat-bit mode at a
    fractional budget).
    """
    if mode in ("uniform", "joint") and avg_bits not in UNIFORM_AVAILABLE_WIDTHS:
        return None

    reader, _ = load_model_and_tokenizer(MODEL_ID, DEVICE)   # own fresh model
    finetune = mode in ("joint", "adaptive_joint", "kl0")
    lambda_kl = 0.0 if mode == "kl0" else LAMBDA_KL

    if mode in ("uniform", "joint"):
        bits_fn = lambda i: int(avg_bits)
    else:
        assignment = jab_assignment_for(avg_bits)
        bits_fn = lambda i: assignment[layer_name(i)]

    cache = {}

    def eval_fn():
        cache["ds_evals"] = eval_all(reader)
        return cache["ds_evals"]["wikitext2"][0]

    _, notes, metrics = pass_quantize_eval(
        reader, config, ATTN, calib_ids, eval_fn, bits_fn=bits_fn, finetune=finetune,
        lambda_kl=lambda_kl, layer_indices=JOINT_LAYERS, label=f"{mode} @ {avg_bits} avg bits",
        return_notes=True, collect_metrics=True, metric_eval_chunks=metric_eval_chunks)

    quant_mean = sum(metrics["quant_error"].values()) / len(metrics["quant_error"])
    rows = []
    for ds_name, (ppl, acc) in cache["ds_evals"].items():
        recon = metrics["attn_recon"].get(ds_name, {})
        recon_mean = sum(recon.values()) / len(recon) if recon else float("nan")
        rows.append({"mode": mode, "avg_bits": avg_bits, "eval_dataset": ds_name,
                     "perplexity": ppl, "accuracy": acc,
                     "quant_error_mean": quant_mean, "attn_recon_error_mean": recon_mean})
    return rows, metrics


for mode in MODES:
    for avg_bits in SWEEP_BITS:
        out = run_arm(mode, avg_bits)
        if out is None:
            for ds_name in EVAL_TEXTS:
                results_rows.append({"mode": mode, "avg_bits": avg_bits, "eval_dataset": ds_name,
                                     "perplexity": None, "accuracy": None,
                                     "quant_error_mean": None, "attn_recon_error_mean": None})
            print(f"{mode} @ {avg_bits}: -- (flat-bit mode undefined at a fractional budget)")
            continue
        rows, per_layer = out
        results_rows.extend(rows)
        per_layer_store[f"{mode}_{avg_bits}"] = per_layer


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


=== pass: uniform @ 3 avg bits ===
  layer  0/12: 3 bits
  layer  1/12: 3 bits
  layer  2/12: 3 bits
  layer  3/12: 3 bits
  layer  4/12: 3 bits
  layer  5/12: 3 bits
  layer  6/12: 3 bits
  layer  7/12: 3 bits
  layer  8/12: 3 bits
  layer  9/12: 3 bits
  layer 10/12: 3 bits
  layer 11/12: 3 bits
    [vram after 'uniform @ 3 avg bits': 1.05 GB now, 1.83 GB peak this pass]
=== uniform @ 3 avg bits: perplexity 39.338 ===
uniform @ 3.5: -- (flat-bit mode undefined at a fractional budget)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


=== pass: uniform @ 4 avg bits ===
  layer  0/12: 4 bits
  layer  1/12: 4 bits
  layer  2/12: 4 bits
  layer  3/12: 4 bits
  layer  4/12: 4 bits
  layer  5/12: 4 bits
  layer  6/12: 4 bits
  layer  7/12: 4 bits
  layer  8/12: 4 bits
  layer  9/12: 4 bits
  layer 10/12: 4 bits
  layer 11/12: 4 bits
    [vram after 'uniform @ 4 avg bits': 1.06 GB now, 1.84 GB peak this pass]
=== uniform @ 4 avg bits: perplexity 26.536 ===
uniform @ 4.5: -- (flat-bit mode undefined at a fractional budget)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...

  avg_bits=4: budget 48.0, used 48.0 (4.00 avg), sensitivity 6.2327e+08

=== pass: kl0 @ 4 avg bits ===
  layer  0/12: 4 bits, FT 0.000387 -> 0.000145, flip rate 0.168
  layer  1/12: 4 bits, FT 0.000901 -> 0.000324, flip rate 0.207
  layer  2/12: 4 bits, FT 0.002508 -> 0.000819, flip rate 0.185
  layer  3/12: 4 bits, FT 0.003471 -> 0.001547, flip rate 0.166
  layer  4/12: 4 bits, FT 0.002943 -> 0.001426, flip rate 0.187
  layer  5/12: 4 bits, FT 0.006562 -> 0.002684, flip rate 0.185
  layer  6/12: 4 bits, FT 0.006702 -> 0.002385, flip rate 0.203
  layer  7/12: 4 bits, FT 0.008238 -> 0.003111, flip rate 0.186
  layer  8/12: 4 bits, FT 0.007842 -> 0.003534, flip rate 0.205
  layer  9/12: 4 bits, FT 0.006450 -> 0.004053, flip rate 0.215
  layer 10/12: 4 bits, FT 0.006083 -> 0.006083, flip rate 0.000
  layer 11/12: 4 bits, FT 0.007971 -> 0.007971, flip rate 0.000
    [vram after 'kl0 @ 4 avg bits': 1.09 GB now, 1.87 GB peak this pass]
=== kl0 @ 4 avg bits: perplexity 26.865 ===


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  avg_bits=4.5: budget 54.0, used 52.0 (4.33 avg), sensitivity 4.6197e+08

=== pass: kl0 @ 4.5 avg bits ===
  layer  0/12: 4 bits, FT 0.000387 -> 0.000145, flip rate 0.168
  layer  1/12: 4 bits, FT 0.000901 -> 0.000324, flip rate 0.207
  layer  2/12: 4 bits, FT 0.002508 -> 0.000819, flip rate 0.185
  layer  3/12: 8 bits, FT 0.000017 -> 0.000010, flip rate 0.178
  layer  4/12: 4 bits, FT 0.002722 -> 0.001444, flip rate 0.189
  layer  5/12: 4 bits, FT 0.007014 -> 0.002532, flip rate 0.178
  layer  6/12: 4 bits, FT 0.006903 -> 0.002356, flip rate 0.201
  layer  7/12: 4 bits, FT 0.008862 -> 0.003059, flip rate 0.190
  layer  8/12: 4 bits, FT 0.008668 -> 0.003712, flip rate 0.195
  layer  9/12: 4 bits, FT 0.006242 -> 0.003790, flip rate 0.205
  layer 10/12: 4 bits, FT 0.006982 -> 0.006982, flip rate 0.000
  layer 11/12: 4 bits, FT 0.008554 -> 0.008554, flip rate 0.000
    [vram after 'kl0 @ 4.5 avg bits': 1.09 GB now, 1.87 GB peak this pass]
=== kl0 @ 4.5 avg bits: perplexity 26.348 ===


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  avg_bits=6: budget 72.0, used 72.0 (6.00 avg), sensitivity 1.5643e+08

=== pass: kl0 @ 6 avg bits ===
  layer  0/12: 4 bits, FT 0.000387 -> 0.000145, flip rate 0.168
  layer  1/12: 4 bits, FT 0.000901 -> 0.000324, flip rate 0.207
  layer  2/12: 8 bits, FT 0.000012 -> 0.000006, flip rate 0.170
  layer  3/12: 8 bits, FT 0.000017 -> 0.000010, flip rate 0.175
  layer  4/12: 8 bits, FT 0.000015 -> 0.000008, flip rate 0.184
  layer  5/12: 8 bits, FT 0.000041 -> 0.000015, flip rate 0.201
  layer  6/12: 8 bits, FT 0.000040 -> 0.000012, flip rate 0.213
  layer  7/12: 8 bits, FT 0.000042 -> 0.000015, flip rate 0.210
  layer  8/12: 4 bits, FT 0.008541 -> 0.003299, flip rate 0.188
  layer  9/12: 4 bits, FT 0.006384 -> 0.003785, flip rate 0.205
  layer 10/12: 4 bits, FT 0.006797 -> 0.006124, flip rate 0.207
  layer 11/12: 4 bits, FT 0.008868 -> 0.008868, flip rate 0.000
    [vram after 'kl0 @ 6 avg bits': 1.09 GB now, 1.87 GB peak this pass]
=== kl0 @ 6 avg bits: perplexity 25.353 ===


## 13. Results: tidy table, `results.csv`, `results_per_layer.json`

In [16]:
FIELDNAMES = ["mode", "avg_bits", "eval_dataset", "perplexity", "accuracy",
             "quant_error_mean", "attn_recon_error_mean"]


def fmt(v, spec):
    return "--" if v is None else format(v, spec)


print(f"{MODEL_ID}  ({ATTN.n_layers} layers, hidden {ATTN.hidden_size}, {ATTN.num_heads} heads)")
print(f"\n{'mode':<16}{'avg_bits':>10}{'eval_dataset':>14}{'perplexity':>13}{'accuracy':>11}"
      f"{'quant_err':>12}{'attn_err':>12}")
for r in results_rows:
    avg_bits_str = "--" if r["avg_bits"] is None else str(r["avg_bits"])
    print(f"{r['mode']:<16}{avg_bits_str:>10}{r['eval_dataset']:>14}"
          f"{fmt(r['perplexity'], '.3f'):>13}{fmt(r['accuracy'], '.4f'):>11}"
          f"{fmt(r['quant_error_mean'], '.4f'):>12}{fmt(r['attn_recon_error_mean'], '.4f'):>12}")

with open("results.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
    writer.writeheader()
    writer.writerows(results_rows)
print("\nSaved results.csv")

with open("results_per_layer.json", "w") as f:
    json.dump(per_layer_store, f, indent=2)
print("Saved results_per_layer.json "
      f"({len(per_layer_store)} arms, each with per-layer quant_error + attn_recon vectors)")


gpt2  (12 layers, hidden 768, 12 heads)

mode              avg_bits  eval_dataset   perplexity   accuracy   quant_err    attn_err
fp32                    --     wikitext2       24.357     0.4148      0.0000      0.0000
fp32                    --            c4       32.250     0.3779      0.0000      0.0000
uniform                  3     wikitext2       39.338     0.3612      0.3753      0.4711
uniform                  3            c4       49.495     0.3339      0.3753      0.4621
uniform                3.5     wikitext2           --         --          --          --
uniform                3.5            c4           --         --          --          --
uniform                  4     wikitext2       26.536     0.4038      0.1739      0.2382
uniform                  4            c4       34.701     0.3692      0.1739      0.2370
uniform                4.5     wikitext2           --         --          --          --
uniform                4.5            c4           --         --     